# 1장. AI와 함께하는 데이터 분석의 시작

LLM 기반 데이터 분석 실무 입문 과정의 실습 노트북입니다.


## 학습 목표

- 주요 개념과 분석 흐름을 이해합니다.
- 제공된 실습 데이터를 불러와 기본 구조를 확인합니다.
- LLM을 분석 보조 도구로 활용하는 방법을 익힙니다.


## 실습 배경

이번 장의 내용을 실습하면서 데이터 로드, 탐색, 전처리, 시각화의 기본 흐름을 확인합니다.


In [1]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

DATA_DIR = Path('../data/raw')
sns.set_theme(style='whitegrid')


## 데이터 불러오기 기본 설정

샘플 데이터를 불러오기 위한 기본 설정입니다. 필요에 따라 아래 코드를 수정해 실행합니다.


In [4]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("../data/raw")

orders = pd.read_csv(DATA_DIR / "orders.csv")
order_items = pd.read_csv(DATA_DIR / "order_items.csv")
products = pd.read_csv(DATA_DIR / "products.csv")

## 실습 진행

이제 데이터를 직접 다루며 간단한 분석 결과를 확인합니다. 필요한 코드를 자유롭게 추가하세요.


#### 1. 컬럼이 실제로 존재하는지 확인
 - orders: order_id, order_date, order_status
 - order_items: order_id, product_id, quantity, unit_price
 - products: product_id, product_name


In [7]:
#1. 컬럼이 실제로 존재하는지 확인
from pathlib import Path
import pandas as pd

DATA_DIR = Path("../data/raw")

orders = pd.read_csv(DATA_DIR / "orders.csv")
order_items = pd.read_csv(DATA_DIR / "order_items.csv")
products = pd.read_csv(DATA_DIR / "products.csv")

print(orders.columns.tolist())
print(order_items.columns.tolist())
print(products.columns.tolist())

['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status']
['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price']
['product_id', 'product_name', 'category', 'price']


#### 2. 분석 조건에 맞는 값이 실제로 있는지 확인
 - completed 상태가 실제로 존재하는가?
 - 2026년 3월부터 8월까지의 주문이 있는가?
 - 날짜가 정상적으로 인식되는가?

In [8]:
#2. 분석 조건에 맞는 값이 실제로 있는지 확인
print(orders["order_status"].value_counts())
print(orders["order_date"].min())
print(orders["order_date"].max())

order_status
completed    184
cancelled     64
refunded      52
Name: count, dtype: int64
2025-09-08
2026-09-07


#### 3. 세 파일이 정상적으로 연결되는지 확인
 - product_name의 결측 수가 0이면 product_id 연결이 정상

In [9]:
#3. 세 파일이 정상적으로 연결되는지 확인
orders["order_date"] = pd.to_datetime(orders["order_date"])

completed_orders = orders[
    (orders["order_status"] == "completed")
    & (orders["order_date"] >= "2026-03-01")
    & (orders["order_date"] <= "2026-08-31")
]

merged = completed_orders.merge(
    order_items,
    on="order_id",
    how="inner"
).merge(
    products[["product_id", "product_name"]],
    on="product_id",
    how="left"
)

print(merged.head())
print(merged["product_name"].isna().sum())

   order_id  customer_id order_date payment_method order_status  \
0         1          123 2026-07-07           card    completed   
1         1          123 2026-07-07           card    completed   
2         1          123 2026-07-07           card    completed   
3         1          123 2026-07-07           card    completed   
4         6           87 2026-05-21      naver_pay    completed   

   order_item_id  product_id  quantity  unit_price product_name  
0              1         100         3      102000    도서 상품 100  
1              2          87         5       25000    도서 상품 087  
2              3           7         3      142000    도서 상품 007  
3              4           9         3      193000   스포츠 상품 009  
4             13          83         3       24000  전자기기 상품 083  
0


#### 4. 판매 금액을 직접 계산해보기
 - 상품별 판매 수량과 판매 금액이 계산되는가?

In [10]:
# 4. 판매 금액을 직접 계산해보기
merged["sales_amount"] = (
    merged["quantity"] * merged["unit_price"]
)

product_sales = (
    merged.groupby(["product_id", "product_name"], as_index=False)
    .agg(
        total_quantity=("quantity", "sum"),
        total_sales=("sales_amount", "sum")
    )
    .sort_values("total_sales", ascending=False)
)

print(product_sales.head(10))

    product_id product_name  total_quantity  total_sales
9           12    식품 상품 012              17      2975000
6            9   스포츠 상품 009              15      2895000
33          41   스포츠 상품 041              17      2771000
15          18   스포츠 상품 018              23      2507000
62          71  전자기기 상품 071              15      2415000
64          73    도서 상품 073              20      2280000
40          49  전자기기 상품 049              13      2093000
30          38  생활용품 상품 038              12      2076000
18          22  생활용품 상품 022              18      2016000
80          92    식품 상품 092              17      1887000


#### 5. 이상값 확인
현실에서 불가능한 수치 등을 확인
 - 0 이하 수량
 - 마이너스 단가

In [11]:
# 5. 이상값 확인
print(merged[["quantity", "unit_price"]].describe())

print("결측치")
print(merged[["order_id", "product_id", "quantity", "unit_price"]].isna().sum())

print("0 이하 수량")
print((merged["quantity"] <= 0).sum())

print("0 이하 단가")
print((merged["unit_price"] <= 0).sum())

         quantity     unit_price
count  251.000000     251.000000
mean     2.908367  103103.585657
std      1.412646   57084.824841
min      1.000000    5000.000000
25%      2.000000   55000.000000
50%      3.000000  109000.000000
75%      4.000000  161000.000000
max      5.000000  200000.000000
결측치
order_id      0
product_id    0
quantity      0
unit_price    0
dtype: int64
0 이하 수량
0
0 이하 단가
0


## LLM 프롬프트 활용 예시

이번에는 LLM에게 분석 질문이나 코드 초안을 요청하는 연습을 할 수 있습니다. API Key는 노트북에 직접 입력하지 않습니다.

```text
분석 목적을 설명하고 필요한 코드를 요청하세요.
```


## 실습 과제

1. 이번 장에서 배운 내용을 바탕으로 분석 질문 3개를 작성합니다.
2. 그중 하나를 pandas 코드로 구현합니다.
3. 결과를 표 또는 그래프로 확인하고, LLM 도움을 받아 해석합니다.
